# Avance 2 · SSC — Etapa 2: perfilamiento y validación
**Equipo A&C · Proyecto de Inteligencia de Negocios**

**Pregunta de negocio:** ¿En qué alcaldías y horarios de la CDMX se concentra la mayor frecuencia de hechos de tránsito y en cuáles se observa mayor severidad registrada (personas lesionadas y fallecidas)?

En este notebook revisamos la calidad de los **cinco archivos SSC delimitados en la Etapa 1**. No modificamos los datos crudos, no eliminamos filas ni asumimos que los folios repetidos sean duplicados. Documentamos hallazgos y posibles reglas para la Etapa 3.

**Instrucciones:** primero ejecutamos el notebook `01_etapa1_delimitacion_SSC_CORREGIDO.ipynb` para generar los CSV de la Etapa 1. Después abrimos este notebook y elegimos **Run → Run All Cells**. No necesitamos `funciones_calidad.py`.

## 1. Bibliotecas y ubicación de los resultados de la Etapa 1
Buscamos la carpeta `resultados_SSC/etapa1` junto a nuestro notebook o dentro de la carpeta personal de Jupyter. Si hay varias copias, indicamos la correcta en `RUTA_ETAPA1_MANUAL`. **Nunca mezclamos archivos de carpetas distintas.**

In [1]:
from pathlib import Path
import json
import re
import unicodedata
import pandas as pd
import numpy as np
from IPython.display import display

RUTA_ETAPA1_MANUAL = None  # Ejemplo: Path('/home/usuario/proyecto/resultados_SSC/etapa1')

NOMBRES_BASE = [
    'HechosTransito_SSC',
    'PersonasLyFporEdad_SSC',
    'PersonasLyFporSexo_SSC',
    'PersonasLyFporTipoPersona_SSC',
    'VehiculosInvolucrados_SSC'
]

def archivos_validos(carpeta):
    carpeta = Path(carpeta)
    return [carpeta / (base + '_etapa1.csv') for base in NOMBRES_BASE]

if RUTA_ETAPA1_MANUAL is not None:
    CARPETA_ETAPA1 = Path(RUTA_ETAPA1_MANUAL).expanduser().resolve()
else:
    actual = Path.cwd().resolve()
    candidatos = []
    for padre in [actual, *actual.parents]:
        candidatos.append(padre / 'resultados_SSC' / 'etapa1')
    candidatos = [p for p in candidatos if all(x.is_file() for x in archivos_validos(p))]
    if not candidatos:
        candidatos = [
            p for p in Path.home().rglob('resultados_SSC/etapa1')
            if all(x.is_file() for x in archivos_validos(p))
        ]
    candidatos = sorted(set(p.resolve() for p in candidatos))
    if not candidatos:
        raise FileNotFoundError(
            'No encontramos los cinco CSV delimitados. Ejecuta primero la Etapa 1 '
            'o configura RUTA_ETAPA1_MANUAL con la carpeta correcta.'
        )
    if len(candidatos) > 1:
        print('AVISO: encontramos varias carpetas completas. Usaremos la primera:')
        for p in candidatos:
            print(' -', p)
        print('Para elegir otra, cambia RUTA_ETAPA1_MANUAL y vuelve a ejecutar.')
    CARPETA_ETAPA1 = candidatos[0]

ARCHIVOS = archivos_validos(CARPETA_ETAPA1)
faltantes = [p.name for p in ARCHIVOS if not p.is_file()]
assert not faltantes, 'Faltan archivos: ' + ', '.join(faltantes)

DESTINO_ETAPA2 = CARPETA_ETAPA1.parent / 'etapa2'
DESTINO_ETAPA2.mkdir(parents=True, exist_ok=True)

print('Carpeta de entrada:', CARPETA_ETAPA1)
print('Carpeta de evidencias:', DESTINO_ETAPA2)
print('Archivos aptos:', len(ARCHIVOS))
for p in ARCHIVOS:
    print(' -', p.name)
print('Versión de pandas:', pd.__version__)

Carpeta de entrada: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_SSC/etapa1
Carpeta de evidencias: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_SSC/etapa2
Archivos aptos: 5
 - HechosTransito_SSC_etapa1.csv
 - PersonasLyFporEdad_SSC_etapa1.csv
 - PersonasLyFporSexo_SSC_etapa1.csv
 - PersonasLyFporTipoPersona_SSC_etapa1.csv
 - VehiculosInvolucrados_SSC_etapa1.csv
Versión de pandas: 1.4.2


## 2. Contrato de lectura e inventario
Leemos los cinco CSV **como texto** (`dtype=str`) para no alterar folios ni códigos. Verificamos que los archivos tengan las columnas esperadas y guardamos sus dimensiones. Recordamos que las tablas tienen diferentes unidades de observación: hechos, desgloses de personas y vehículos.

In [2]:
TABLAS = {}
inventario = []
for ruta in ARCHIVOS:
    nombre = ruta.stem.replace('_etapa1', '')
    df = pd.read_csv(ruta, dtype=str, encoding='utf-8-sig', low_memory=False)
    TABLAS[nombre] = df
    inventario.append({
        'archivo': ruta.name,
        'tabla': nombre,
        'filas': len(df),
        'columnas': len(df.columns),
        'tiene_folio': 'folio' in df.columns,
        'tiene_fecha_evento': 'fecha_evento' in df.columns,
        'tiene_alcaldia': 'alcaldia' in df.columns,
        'tiene_alcaldia_normalizada': 'alcaldia_normalizada' in df.columns
    })
inventario_df = pd.DataFrame(inventario)
display(inventario_df)
inventario_df.to_csv(DESTINO_ETAPA2 / 'inventario_etapa2_SSC.csv', index=False, encoding='utf-8-sig')
assert len(TABLAS) == 5
assert inventario_df[['tiene_fecha_evento', 'tiene_alcaldia']].all().all(), 'Revisar columnas indispensables'

,archivo,tabla,filas,columnas,tiene_folio,tiene_fecha_evento,tiene_alcaldia,tiene_alcaldia_normalizada
0,HechosTransito_SSC_etapa1.csv,HechosTransito_SSC,134078,27,True,True,True,True
1,PersonasLyFporEdad_SSC_etapa1.csv,PersonasLyFporEdad_SSC,158076,24,True,True,True,True
2,PersonasLyFporSexo_SSC_etapa1.csv,PersonasLyFporSexo_SSC,144043,25,True,True,True,True
3,PersonasLyFporTipoPersona_SSC_etapa1.csv,PersonasLyFporTipoPersona_SSC,107794,25,True,True,True,True
4,VehiculosInvolucrados_SSC_etapa1.csv,VehiculosInvolucrados_SSC,215078,27,True,True,True,True


## 3. Exploración inicial: primeras y últimas filas
Inspeccionamos las primeras y últimas tres filas de cada tabla y registramos su esquema. La visualización no cambia los datos.

In [3]:
esquemas = []
for nombre, df in TABLAS.items():
    print('\n' + '=' * 65)
    print(nombre, '|', len(df), 'filas |', len(df.columns), 'columnas')
    print('PRIMERAS 3 FILAS')
    display(df.head(3))
    print('ÚLTIMAS 3 FILAS')
    display(df.tail(3))
    for col in df.columns:
        esquemas.append({'tabla': nombre, 'columna': col, 'tipo_lectura': str(df[col].dtype)})
pd.DataFrame(esquemas).to_csv(
    DESTINO_ETAPA2 / 'esquema_columnas_SSC.csv', index=False, encoding='utf-8-sig'
)


HechosTransito_SSC | 134078 filas | 27 columnas
PRIMERAS 3 FILAS


,fecha_evento,hora_evento,tipo_evento,fecha_captura,folio,latitud,longitud,punto_1,punto_2,colonia,...,sentido_de_circulacion,dia,prioridad,origen,unidad_medica_de_apoyo,matricula_unidad_medica,trasladado_lesionados,personas_fallecidas,personas_lesionadas,alcaldia_normalizada
0,2020-04-06,12:50:00,CHOQUE,2020-04-17,BJ/200406/03499,19.368116,-99.142903,EJE 7 SUR,ANTILLAS,PORTALES NTE,...,P-O,Lunes,BAJA,RADIO,PC,NaN,NO,0,1,BENITO JUAREZ
1,2020-04-06,18:31:00,CHOQUE,2020-04-17,C5/200406/05748,19.301142,-99.115521,CALZ DEL HUESO,RANCHO COLORADO,COAPA STA CECILIA,...,O-P,Lunes,BAJA,911 CDMX,ERUM,NaN,NO,0,1,COYOACAN
2,2020-04-06,18:39:00,CHOQUE,2020-04-17,C5/200406/05802,19.476843,-99.092207,EJE 5 NTE,AV GRAN CANAL DEL DESAGUE,JOSE MA MORELOS Y PAVON,...,P-O,Lunes,BAJA,911 CDMX,PARTICULAR,NaN,NO,0,1,GUSTAVO A MADERO


ÚLTIMAS 3 FILAS


,fecha_evento,hora_evento,tipo_evento,fecha_captura,folio,latitud,longitud,punto_1,punto_2,colonia,...,sentido_de_circulacion,dia,prioridad,origen,unidad_medica_de_apoyo,matricula_unidad_medica,trasladado_lesionados,personas_fallecidas,personas_lesionadas,alcaldia_normalizada
134075,2023-12-31,NaN,CHOQUE,2024-02-12,C5/20231231/01445,19.40747,-99.19997,AV CONSTITUYENTES,GRAL JOSE MA MENDIVIL,DANIEL GARZA,...,NO-SP,Domingo,MEDIA,LLAMADA DEL 911,PARTICULAR,PB03,SI,0,2,MIGUEL HIDALGO
134076,2023-12-31,NaN,CHOQUE,2024-02-12,C5/20231231/04116,19.439667,-99.207823,PRESA LA ANGOSTURA,PRESA COINTZIO,IRRIGACION,...,N-S,Domingo,BAJA,LLAMADA DEL 911,PC,6,NO,0,1,MIGUEL HIDALGO
134077,2023-12-31,NaN,CHOQUE,2024-02-08,C5/20231231/00301,19.391144,-99.086187,EJE 4 SUR,AV PDTE PLUTARCO ELIAS CALLES,EL RODEO,...,NP-SO,Domingo,ALTA,LLAMADA DEL 911,CCO/CRUZ ROJA,23,NO,1,2,IZTACALCO



PersonasLyFporEdad_SSC | 158076 filas | 24 columnas
PRIMERAS 3 FILAS


...

,fecha_evento,hora_evento,tipo_evento,fecha_captura,folio,latitud,longitud,punto_1,punto_2,colonia,...,interseccion_semaforizada,clasificacion_de_la_vialidad,sentido_de_circulacion,dia,prioridad,origen,tipo_persona,condicion_persona,total,alcaldia_normalizada
107791,2023-12-31,NaN,CHOQUE,2024-02-08,C5/20231231/00301,19.391144,-99.086187,EJE 4 SUR,AV PDTE PLUTARCO ELIAS CALLES,EL RODEO,...,SI,EJE VIAL,NP-SO,Domingo,ALTA,LLAMADA DEL 911,Motociclista,fallecida,1,IZTACALCO
107792,2023-12-31,NaN,CHOQUE,2024-02-08,C5/20231231/00301,19.391144,-99.086187,EJE 4 SUR,AV PDTE PLUTARCO ELIAS CALLES,EL RODEO,...,SI,EJE VIAL,NP-SO,Domingo,ALTA,LLAMADA DEL 911,Pasajera,lesionada,1,IZTACALCO
107793,2023-12-31,NaN,CHOQUE,2024-02-08,C5/20231231/00301,19.391144,-99.086187,EJE 4 SUR,AV PDTE PLUTARCO ELIAS CALLES,EL RODEO,...,SI,EJE VIAL,NP-SO,Domingo,ALTA,LLAMADA DEL 911,Motociclista,lesionada,1,IZTACALCO



VehiculosInvolucrados_SSC | 215078 filas | 27 columnas
PRIMERAS 3 FILAS


,fecha_evento,hora_evento,tipo_evento,fecha_captura,folio,latitud,longitud,punto_1,punto_2,colonia,...,sentido_de_circulacion,dia,prioridad,origen,unidad_medica_de_apoyo,matricula_unidad_medica,trasladado_lesionados,no_vehiculo,tipo_vehiculo,alcaldia_normalizada
0,2018-01-01,23:16:00,ATROPELLADO,2018-03-01,316753,19.437784,-99.140321,EJE CENTRAL,REP DE CUBA,CENTRO,...,NaN,Lunes,BAJA,NaN,CRUZ ROJA,NaN,NO,Vehículo 1,SIN DATO,CUAUHTEMOC
1,2018-01-01,06:11:00,CHOQUE,2018-03-01,328860,19.496948,-99.073225,TIXTLA,SAN JUAN DE LOS LAGOS,SAN FELIPE DE JESUS,...,NaN,Lunes,MEDIA,NaN,ERUM,NaN,SI,Vehículo 1,AUTOMOVIL,GUSTAVO A MADERO
2,2018-01-01,06:11:00,CHOQUE,2018-03-01,328860,19.496948,-99.073225,TIXTLA,SAN JUAN DE LOS LAGOS,SAN FELIPE DE JESUS,...,NaN,Lunes,MEDIA,NaN,ERUM,NaN,SI,Vehículo 2,AUTOMOVIL,GUSTAVO A MADERO


ÚLTIMAS 3 FILAS


,fecha_evento,hora_evento,tipo_evento,fecha_captura,folio,latitud,longitud,punto_1,punto_2,colonia,...,sentido_de_circulacion,dia,prioridad,origen,unidad_medica_de_apoyo,matricula_unidad_medica,trasladado_lesionados,no_vehiculo,tipo_vehiculo,alcaldia_normalizada
215075,2023-12-31,NaN,CHOQUE,2024-02-12,C5/20231231/04116,19.439667,-99.207823,PRESA LA ANGOSTURA,PRESA COINTZIO,IRRIGACION,...,N-S,Domingo,BAJA,LLAMADA DEL 911,PC,6,NO,Vehículo 2,AUTOMOVIL,MIGUEL HIDALGO
215076,2023-12-31,NaN,CHOQUE,2024-02-08,C5/20231231/00301,19.391144,-99.086187,EJE 4 SUR,AV PDTE PLUTARCO ELIAS CALLES,EL RODEO,...,NP-SO,Domingo,ALTA,LLAMADA DEL 911,CCO/CRUZ ROJA,23,NO,Vehículo 1,MOTOCICLETA,IZTACALCO
215077,2023-12-31,NaN,CHOQUE,2024-02-08,C5/20231231/00301,19.391144,-99.086187,EJE 4 SUR,AV PDTE PLUTARCO ELIAS CALLES,EL RODEO,...,NP-SO,Domingo,ALTA,LLAMADA DEL 911,CCO/CRUZ ROJA,23,NO,Vehículo 2,MOTOCICLETA,IZTACALCO


## 4. Perfil de calidad de **todas las columnas**
Medimos nulos reales, textos vacíos, valores distintos y ejemplos. Tratamos cadenas vacías o formadas por espacios como faltantes para el **reporte**, sin sustituirlas en el CSV. Los códigos `SD`, `S/D` y similares se analizan por separado; no los interpretamos automáticamente como valores ausentes de todas las columnas.

In [4]:
def perfil_calidad(df, nombre):
    filas = []
    for col in df.columns:
        s = df[col]
        vacios = s.notna() & s.astype(str).str.strip().eq('')
        faltantes = s.isna() | vacios
        ejemplos = s.loc[~faltantes].drop_duplicates().head(4).tolist()
        filas.append({
            'tabla': nombre,
            'columna': col,
            'tipo_lectura': str(s.dtype),
            'filas': len(df),
            'nulos_pandas': int(s.isna().sum()),
            'vacios_o_espacios': int(vacios.sum()),
            'faltantes_totales': int(faltantes.sum()),
            'porcentaje_faltantes': round(100 * faltantes.mean(), 2) if len(df) else 0.0,
            'valores_distintos_no_nulos': int(s.nunique(dropna=True)),
            'ejemplos': ' | '.join(map(str, ejemplos))
        })
    return pd.DataFrame(filas)

perfiles = [perfil_calidad(df, nombre) for nombre, df in TABLAS.items()]
perfil_df = pd.concat(perfiles, ignore_index=True)
perfil_df.to_csv(DESTINO_ETAPA2 / 'perfil_columnas_SSC.csv', index=False, encoding='utf-8-sig')
print('Columnas perfiladas:', len(perfil_df))
display(perfil_df.sort_values(['tabla', 'porcentaje_faltantes'], ascending=[True, False]))

Columnas perfiladas: 128


,tabla,columna,tipo_lectura,filas,nulos_pandas,vacios_o_espacios,faltantes_totales,porcentaje_faltantes,valores_distintos_no_nulos,ejemplos
22,HechosTransito_SSC,matricula_unidad_medica,object,134078,82187,0,82187,61.30,4326,SD | 9618 | MXPS19 | MXPR17
13,HechosTransito_SSC,unidad_a_cargo,object,134078,65746,0,65746,49.04,7666,MX164B4 | MX337S2 | MX329N2 | SD
3,HechosTransito_SSC,fecha_captura,object,134078,38200,0,38200,28.49,1608,2020-04-17 | 2020-04-18 | 2020-04-20 | 2020-04-23
15,HechosTransito_SSC,interseccion_semaforizada,object,134078,35042,0,35042,26.14,3,NO | SI | N
16,HechosTransito_SSC,clasificacion_de_la_vialidad,object,134078,35042,0,35042,26.14,8,EJE VIAL | VIA PRIMARIA | VAC ANULAR | VIA SEC...
...,...,...,...,...,...,...,...,...,...,...
120,VehiculosInvolucrados_SSC,prioridad,object,215078,0,0,0,0.00,3,BAJA | MEDIA | ALTA
122,VehiculosInvolucrados_SSC,unidad_medica_de_apoyo,object,215078,3,0,3,0.00,2434,CRUZ ROJA | ERUM | CCO | ERUM/CCO
124,VehiculosInvolucrados_SSC,trasladado_lesionados,object,215078,3,0,3,0.00,2,NO | SI
125,VehiculosInvolucrados_SSC,no_vehiculo,object,215078,0,0,0,0.00,3,Vehículo 1 | Vehículo 2 | Vehículo 3


## 5. Frecuencias, valores faltantes y categorías inesperadas
Para **cada columna**, registramos las diez categorías más frecuentes, incluyendo valores nulos. Esto nos ayuda a identificar categorías `SD`, códigos, errores de escritura y valores poco frecuentes. El reporte completo se guarda como CSV.

In [5]:
frecuencias = []
for nombre, df in TABLAS.items():
    for col in df.columns:
        conteos = df[col].value_counts(dropna=False).head(10)
        for puesto, (valor, cantidad) in enumerate(conteos.items(), start=1):
            frecuencias.append({
                'tabla': nombre, 'columna': col, 'puesto': puesto,
                'valor': '<NULO>' if pd.isna(valor) else str(valor),
                'cantidad': int(cantidad),
                'porcentaje_tabla': round(100 * cantidad / len(df), 2) if len(df) else 0.0
            })
frecuencias_df = pd.DataFrame(frecuencias)
frecuencias_df.to_csv(DESTINO_ETAPA2 / 'frecuencias_top10_SSC.csv',
                      index=False, encoding='utf-8-sig')
print('Frecuencias registradas:', len(frecuencias_df))
display(frecuencias_df.loc[
    frecuencias_df['columna'].isin(['alcaldia', 'alcaldia_normalizada', 'folio'])
].head(40))

Frecuencias registradas: 1108


,tabla,columna,puesto,valor,cantidad,porcentaje_tabla
36,HechosTransito_SSC,folio,1,SD,7288,5.44
37,HechosTransito_SSC,folio,2,617475,5,0.00
38,HechosTransito_SSC,folio,3,543821,3,0.00
39,HechosTransito_SSC,folio,4,437795,3,0.00
40,HechosTransito_SSC,folio,5,566525,3,0.00
41,HechosTransito_SSC,folio,6,574468,3,0.00
42,HechosTransito_SSC,folio,7,576263,3,0.00
43,HechosTransito_SSC,folio,8,593455,3,0.00
44,HechosTransito_SSC,folio,9,605767,3,0.00
45,HechosTransito_SSC,folio,10,625955,3,0.00


## 6. Verificación temporal y geográfica
Comprobamos que las fechas puedan interpretarse como `AAAA-MM-DD`, que sean desde 2018 y que la columna auxiliar contenga únicamente las 16 alcaldías. También revisamos fechas posteriores al día de ejecución como **alertas**, sin borrarlas automáticamente.

In [7]:
ALCALDIAS = {
    'ALVARO OBREGON', 'AZCAPOTZALCO', 'BENITO JUAREZ', 'COYOACAN',
    'CUAJIMALPA DE MORELOS', 'CUAUHTEMOC', 'GUSTAVO A MADERO',
    'IZTACALCO', 'IZTAPALAPA', 'LA MAGDALENA CONTRERAS',
    'MIGUEL HIDALGO', 'MILPA ALTA', 'TLAHUAC', 'TLALPAN',
    'VENUSTIANO CARRANZA', 'XOCHIMILCO'
}
def normalizar(valor):
    if pd.isna(valor):
        return ''
    texto = unicodedata.normalize('NFKD', str(valor).strip().upper())
    return ' '.join(''.join(c for c in texto if not unicodedata.combining(c)).split())

validaciones = []
hoy = pd.Timestamp.today().normalize()
for nombre, df in TABLAS.items():
    fechas = pd.to_datetime(df['fecha_evento'], format='%Y-%m-%d', errors='coerce')
    if 'alcaldia_normalizada' in df:
        alcaldias = df['alcaldia_normalizada'].map(normalizar)
    else:
        alcaldias = df['alcaldia'].map(normalizar)
    validaciones.append({
        'tabla': nombre,
        'filas': len(df),
        'fechas_invalidas': int(fechas.isna().sum()),
        'fechas_antes_2018': int((fechas.notna() & (fechas < pd.Timestamp('2018-01-01'))).sum()),
        'fechas_futuras_alerta': int((fechas.notna() & (fechas > hoy)).sum()),
        'alcaldias_no_validas': int((~alcaldias.isin(ALCALDIAS)).sum()),
        'anios_encontrados': ', '.join(map(str, sorted(fechas.dropna().dt.year.unique().tolist())))
    })
validaciones_df = pd.DataFrame(validaciones)
display(validaciones_df)
validaciones_df.to_csv(DESTINO_ETAPA2 / 'validacion_temporal_geografica_SSC.csv',
                       index=False, encoding='utf-8-sig')
assert (validaciones_df['fechas_invalidas'] == 0).all(), 'Existen fechas inválidas: revisar evidencia'
assert (validaciones_df['fechas_antes_2018'] == 0).all(), 'Existen fechas anteriores a 2018'
assert (validaciones_df['alcaldias_no_validas'] == 0).all(), 'Existen alcaldías no válidas'

,tabla,filas,fechas_invalidas,fechas_antes_2018,fechas_futuras_alerta,alcaldias_no_validas,anios_encontrados
0,HechosTransito_SSC,134078,0,0,0,0,"2018, 2019, 2020, 2021, 2022, 2023"
1,PersonasLyFporEdad_SSC,158076,0,0,0,0,"2018, 2019, 2020, 2021, 2022, 2023"
2,PersonasLyFporSexo_SSC,144043,0,0,0,0,"2018, 2019, 2020, 2021, 2022, 2023"
3,PersonasLyFporTipoPersona_SSC,107794,0,0,0,0,"2020, 2021, 2022, 2023"
4,VehiculosInvolucrados_SSC,215078,0,0,0,0,"2018, 2019, 2020, 2021, 2022, 2023"


## 7. Folios repetidos y valores `SD`
Contamos folios únicos, vacíos, `SD` y folios que aparecen más de una vez. Un folio puede aparecer repetido porque describe **varias personas o vehículos del mismo evento**. Por tanto, **un folio repetido no equivale a un duplicado eliminable**. Guardamos ejemplos para investigar.

In [8]:
folios_resumen = []
folios_repetidos = []
for nombre, df in TABLAS.items():
    if 'folio' not in df.columns:
        folios_resumen.append({'tabla': nombre, 'tiene_folio': False})
        continue
    s = df['folio']
    normal = s.fillna('').astype(str).str.strip().str.upper()
    sin_folio = normal.eq('')
    folio_sd = normal.isin(['SD', 'S/D', 'SIN DATO', 'SIN DATOS', 'N/D', 'ND'])
    utilizables = normal.loc[~(sin_folio | folio_sd)]
    conteos = utilizables.value_counts()
    repetidos = conteos.loc[conteos > 1]
    folios_resumen.append({
        'tabla': nombre, 'tiene_folio': True,
        'folios_vacios': int(sin_folio.sum()),
        'folios_sd_o_equivalentes': int(folio_sd.sum()),
        'folios_distintos_utilizables': int(conteos.size),
        'folios_utilizables_repetidos': int(repetidos.size),
        'filas_en_folios_repetidos': int(repetidos.sum()),
        'maximo_filas_por_folio': int(conteos.max()) if len(conteos) else 0
    })
    for folio, cantidad in repetidos.head(100).items():
        folios_repetidos.append({'tabla': nombre, 'folio': folio, 'filas': int(cantidad)})
folios_df = pd.DataFrame(folios_resumen)
folios_ejemplos_df = pd.DataFrame(folios_repetidos,
    columns=['tabla', 'folio', 'filas'])
display(folios_df)
folios_df.to_csv(DESTINO_ETAPA2 / 'resumen_folios_SSC.csv', index=False, encoding='utf-8-sig')
folios_ejemplos_df.to_csv(DESTINO_ETAPA2 / 'folios_repetidos_top100_SSC.csv',
                          index=False, encoding='utf-8-sig')

,tabla,tiene_folio,folios_vacios,folios_sd_o_equivalentes,folios_distintos_utilizables,folios_utilizables_repetidos,filas_en_folios_repetidos,maximo_filas_por_folio
0,HechosTransito_SSC,True,1,7288,126470,218,537,5
1,PersonasLyFporEdad_SSC,True,0,8651,126251,16718,39892,25
2,PersonasLyFporSexo_SSC,True,1,7828,126470,9433,19177,6
3,PersonasLyFporTipoPersona_SSC,True,0,17,98998,8553,17332,5
4,VehiculosInvolucrados_SSC,True,2,11797,126470,72779,149588,9


## 8. Duplicados exactos y candidatos para revisión
Buscamos filas **idénticas en todas sus columnas**. Las contamos, pero no eliminamos ninguna: incluso las filas idénticas podrían corresponder a observaciones diferentes según la unidad de registro. También guardamos una muestra de hasta 20 filas repetidas por tabla.

**Regla docente que documentamos:** primero identificamos folios repetidos; si la alcaldía difiere, conservamos los registros. Si coincide, comparamos las filas completas. La decisión final requiere revisar la granularidad y documentar la justificación en la Etapa 3.

In [9]:
duplicados_resumen = []
muestras_duplicados = []
for nombre, df in TABLAS.items():
    # keep=False permite mostrar todos los miembros de un grupo idéntico.
    mascara_grupos = df.duplicated(keep=False)
    mascara_sobrantes = df.duplicated(keep='first')
    duplicados_resumen.append({
        'tabla': nombre,
        'filas': len(df),
        'filas_en_grupos_identicos': int(mascara_grupos.sum()),
        'filas_repetidas_adicionales': int(mascara_sobrantes.sum()),
        'filas_sin_repeticion_adicional': int(len(df) - mascara_sobrantes.sum()),
        'filas_eliminadas_en_etapa2': 0
    })
    muestra = df.loc[mascara_grupos].head(20).copy()
    if not muestra.empty:
        muestra.insert(0, 'tabla', nombre)
        muestras_duplicados.append(muestra)
    # Guardamos por tabla para conservar sus diferentes esquemas.
    muestra.to_csv(DESTINO_ETAPA2 / f'{nombre}_muestra_duplicados_exactos.csv',
                  index=False, encoding='utf-8-sig')

duplicados_df = pd.DataFrame(duplicados_resumen)
display(duplicados_df)
duplicados_df.to_csv(DESTINO_ETAPA2 / 'resumen_duplicados_SSC.csv',
                     index=False, encoding='utf-8-sig')

,tabla,filas,filas_en_grupos_identicos,filas_repetidas_adicionales,filas_sin_repeticion_adicional,filas_eliminadas_en_etapa2
0,HechosTransito_SSC,134078,0,0,134078,0
1,PersonasLyFporEdad_SSC,158076,6486,3498,154578,0
2,PersonasLyFporSexo_SSC,144043,0,0,144043,0
3,PersonasLyFporTipoPersona_SSC,107794,0,0,107794,0
4,VehiculosInvolucrados_SSC,215078,0,0,215078,0


## 9. Revisión de folios que aparecen en distintas alcaldías
Detectamos, **dentro de cada tabla**, folios utilizables asociados a más de una alcaldía. No los eliminamos. Guardamos ejemplos y cantidades para distinguir posibles eventos compartidos, errores de captura o diferencias de granularidad.

In [10]:
revision_alcaldias = []
for nombre, df in TABLAS.items():
    if 'folio' not in df.columns:
        continue
    folio = df['folio'].fillna('').astype(str).str.strip().str.upper()
    valido = ~folio.isin(['', 'SD', 'S/D', 'SIN DATO', 'SIN DATOS', 'N/D', 'ND'])
    alcaldia = df['alcaldia_normalizada'] if 'alcaldia_normalizada' in df else df['alcaldia']
    temporal = pd.DataFrame({
        'folio': folio.loc[valido],
        'alcaldia': alcaldia.loc[valido].fillna('<NULO>').astype(str)
    })
    conteo = temporal.groupby('folio')['alcaldia'].nunique()
    diferentes = conteo[conteo > 1].sort_values(ascending=False)
    for folio_val, num in diferentes.head(100).items():
        ubicaciones = temporal.loc[temporal['folio'] == folio_val, 'alcaldia'].unique()
        revision_alcaldias.append({
            'tabla': nombre, 'folio': folio_val,
            'cantidad_alcaldias': int(num),
            'alcaldias': ' | '.join(map(str, ubicaciones))
        })
revision_alcaldias_df = pd.DataFrame(
    revision_alcaldias, columns=['tabla', 'folio', 'cantidad_alcaldias', 'alcaldias']
)
revision_alcaldias_df.to_csv(
    DESTINO_ETAPA2 / 'folios_multiples_alcaldias_top100_SSC.csv',
    index=False, encoding='utf-8-sig'
)
print('Casos de ejemplo con folios en distintas alcaldías:', len(revision_alcaldias_df))
display(revision_alcaldias_df.head(20))

Casos de ejemplo con folios en distintas alcaldías: 411


,tabla,folio,cantidad_alcaldias,alcaldias
0,HechosTransito_SSC,617475,5,IZTAPALAPA | VENUSTIANO CARRANZA | TLALPAN | G...
1,HechosTransito_SSC,472563,3,GUSTAVO A MADERO | TLALPAN | CUAJIMALPA DE MOR...
2,HechosTransito_SSC,538057,3,IZTACALCO | MIGUEL HIDALGO | COYOACAN
3,HechosTransito_SSC,437795,3,CUAUHTEMOC | COYOACAN | IZTAPALAPA
4,HechosTransito_SSC,437776,3,COYOACAN | BENITO JUAREZ | IZTAPALAPA
5,HechosTransito_SSC,436728,3,IZTAPALAPA | COYOACAN | MIGUEL HIDALGO
6,HechosTransito_SSC,436675,3,IZTAPALAPA | ALVARO OBREGON | GUSTAVO A MADERO
7,HechosTransito_SSC,435043,3,CUAUHTEMOC | GUSTAVO A MADERO | IZTAPALAPA
8,HechosTransito_SSC,517747,3,IZTACALCO | CUAUHTEMOC | MIGUEL HIDALGO
9,HechosTransito_SSC,531078,3,VENUSTIANO CARRANZA | ALVARO OBREGON | TLAHUAC


## 10. Resumen numérico exploratorio
Identificamos columnas cuyos valores no vacíos parecen principalmente numéricos. Calculamos mínimos, máximos, mediana y posibles negativos como **alertas**, sin asumir que cualquier número es una medida de lesionados o fallecidos. Los identificadores y coordenadas requieren interpretación particular.

In [ ]:
numericos = []
for nombre, df in TABLAS.items():
    for col in df.columns:
        s = df[col].dropna().astype(str).str.strip()
        s = s.loc[s.ne('')]
        if s.empty:
            continue
        convertido = pd.to_numeric(s, errors='coerce')
        proporcion = float(convertido.notna().mean())
        if proporcion < 0.95:
            continue
        valores = convertido.dropna()
        numericos.append({
            'tabla': nombre, 'columna': col,
            'porcentaje_convertible': round(100 * proporcion, 2),
            'valores_numericos': len(valores),
            'minimo': valores.min(),
            'mediana': valores.median(),
            'maximo': valores.max(),
            'cantidad_negativos_alerta': int((valores < 0).sum())
        })
numericos_df = pd.DataFrame(numericos, columns=[
    'tabla', 'columna', 'porcentaje_convertible', 'valores_numericos',
    'minimo', 'mediana', 'maximo', 'cantidad_negativos_alerta'
])
display(numericos_df)
numericos_df.to_csv(DESTINO_ETAPA2 / 'perfil_numerico_SSC.csv',
                     index=False, encoding='utf-8-sig')

## 11. Validación de edades: valores incoherentes y extremos

En la tabla `PersonasLyFporEdad_SSC` revisamos la variable de edad antes de usarla para interpretar los resultados. **No suponemos que una edad de 300 años sea correcta ni la sustituimos automáticamente.** Utilizamos el intervalo de **0 a 120 años como criterio operativo de revisión**, sujeto a confirmación con el diccionario de datos de SSC. Distinguimos nulos, códigos como `SD`, textos no convertibles, negativos, valores superiores a 120 y edades dentro del intervalo. Guardamos los casos observados con su folio y demás columnas para trazabilidad. Esta etapa no modifica ni elimina registros.


In [11]:
# VALIDACIÓN DE EDADES (SOLO DIAGNÓSTICO)
import re

edad_df = TABLAS["PersonasLyFporEdad_SSC"]

# Mostramos columnas candidatas para evitar adivinar el nombre.
columnas_candidatas = [c for c in edad_df.columns if "edad" in c.lower()]
print("Columnas relacionadas con edad:", columnas_candidatas)

# Si existe más de una columna de edad, elegimos manualmente la adecuada.
COLUMNA_EDAD = None  # Ejemplo: "edad"; dejar None si hay una sola candidata
if COLUMNA_EDAD is None:
    if len(columnas_candidatas) != 1:
        raise ValueError(
            "Hay cero o varias columnas candidatas. Revisa la lista y asigna "
            "COLUMNA_EDAD con el nombre exacto antes de continuar."
        )
    COLUMNA_EDAD = columnas_candidatas[0]

LIMITE_EDAD_REVISION = 120  # Umbral operativo, no edad corregida
serie_original = edad_df[COLUMNA_EDAD]
texto_edad = serie_original.astype("string").str.strip()
faltante = texto_edad.isna() | texto_edad.eq("").fillna(False)
codigo_sin_dato = texto_edad.str.upper().isin(["SD", "S/D", "SIN DATO", "SIN DATOS", "ND", "N/D", "NA", "N/A"]).fillna(False)
numero_edad = pd.to_numeric(texto_edad, errors="coerce")
no_numerico = (~faltante) & (~codigo_sin_dato) & numero_edad.isna()
negativa = numero_edad.notna() & (numero_edad < 0)
superior_limite = numero_edad.notna() & (numero_edad > LIMITE_EDAD_REVISION)
no_entera = numero_edad.notna() & (numero_edad % 1 != 0)
valida_operativa = numero_edad.notna() & numero_edad.between(0, LIMITE_EDAD_REVISION) & ~no_entera

clasificacion = pd.Series("pendiente", index=edad_df.index, dtype="string")
clasificacion.loc[valida_operativa] = "dentro_rango_operativo"
clasificacion.loc[faltante] = "faltante"
clasificacion.loc[codigo_sin_dato] = "codigo_sin_dato"
clasificacion.loc[no_numerico] = "no_numerica"
clasificacion.loc[negativa] = "negativa"
clasificacion.loc[superior_limite] = "superior_a_120"
clasificacion.loc[no_entera & ~negativa & ~superior_limite] = "decimal_revisar"

resumen_edad = clasificacion.value_counts(dropna=False).rename_axis("categoria").reset_index(name="filas")
resumen_edad["porcentaje"] = (100 * resumen_edad["filas"] / len(edad_df)).round(2)
print("Variable revisada:", COLUMNA_EDAD)
print("Máximo numérico observado:", numero_edad.max())
print("Mínimo numérico observado:", numero_edad.min())
display(resumen_edad)
resumen_edad.to_csv(DESTINO_ETAPA2 / "resumen_validacion_edad_SSC.csv", index=False, encoding="utf-8-sig")

# Evidencia: filas completas, SIN alteraciones, con clasificación y valor numérico auxiliar.
casos_revisar = edad_df.loc[clasificacion.ne("dentro_rango_operativo")].copy()
casos_revisar.insert(0, "categoria_revision_edad", clasificacion.loc[casos_revisar.index].to_numpy())
casos_revisar.insert(1, "edad_interpretada_solo_diagnostico", numero_edad.loc[casos_revisar.index].to_numpy())
casos_revisar.to_csv(DESTINO_ETAPA2 / "casos_edad_para_revision_SSC.csv", index=False, encoding="utf-8-sig")

# Frecuencias de edades extremas: identifica explícitamente edades como 300 o superiores.
extremas = pd.DataFrame({"edad_original": texto_edad.loc[superior_limite]}).value_counts(dropna=False).reset_index(name="filas")
extremas.to_csv(DESTINO_ETAPA2 / "frecuencias_edades_superiores_120_SSC.csv", index=False, encoding="utf-8-sig")
print("\nEdades superiores a 120 (frecuencias):")
display(extremas.head(30))
print("\nMuestra de registros que requieren revisión:")
display(casos_revisar.head(15))
print("\nFilas originales intactas:", len(edad_df))
print("Casos marcados para revisión:", len(casos_revisar))
assert int(resumen_edad["filas"].sum()) == len(edad_df)


Columnas relacionadas con edad: ['edad']
Variable revisada: edad
Máximo numérico observado: 523.0
Mínimo numérico observado: 0.0


,categoria,filas,porcentaje
0,dentro_rango_operativo,144144,91.19
1,faltante,13918,8.8
2,superior_a_120,14,0.01



Edades superiores a 120 (frecuencias):


,edad_original,filas
0,223,3
1,125,1
2,247,1
3,250,1
4,274,1
5,302,1
6,333,1
7,355,1
8,368,1
9,410,1



Muestra de registros que requieren revisión:


,categoria_revision_edad,edad_interpretada_solo_diagnostico,fecha_evento,hora_evento,tipo_evento,fecha_captura,folio,latitud,longitud,punto_1,...,tipo_de_interseccion,interseccion_semaforizada,clasificacion_de_la_vialidad,sentido_de_circulacion,dia,prioridad,origen,condicion_persona,edad,alcaldia_normalizada
5,faltante,NaN,2020-04-06,15:31:00,DERRAPADO,2020-04-17,C5/200406/04540,19.403471,-99.192495,ANILLO PERIFERICO,...,T,NO,VAC ANULAR,S-N,Lunes,MEDIA,911 CDMX,lesionada,NaN,MIGUEL HIDALGO
6,faltante,NaN,2020-04-07,10:19:00,ATROPELLADO,2020-04-18,C5/200407/02113,19.47042,-99.11949,AV HENRY FORD,...,T,NO,VIA SECUNDARIA,NP-SO,Martes,MEDIA,RADIO,lesionada,NaN,GUSTAVO A MADERO
7,faltante,NaN,2020-04-07,19:05:00,ATROPELLADO,2020-04-18,GA/200407/05427,19.466033,-99.118281,EJE 1 OTE,...,CRUZ,SI,EJE VIAL,NO-SP,Martes,BAJA,BOTON DE AUXILIO,lesionada,NaN,GUSTAVO A MADERO
16,faltante,NaN,2020-04-07,12:17:00,DERRAPADO,2020-04-18,BJ/200407/02054,19.361559,-99.142779,CALZ DE TLALPAN,...,T,NO,VAC RADIAL,S-N,Martes,BAJA,RADIO,lesionada,NaN,BENITO JUAREZ
17,faltante,NaN,2020-04-07,12:17:00,DERRAPADO,2020-04-18,BJ/200407/02054,19.361559,-99.142779,CALZ DE TLALPAN,...,T,NO,VAC RADIAL,S-N,Martes,BAJA,RADIO,lesionada,NaN,BENITO JUAREZ
21,faltante,NaN,2020-04-08,07:48:00,DERRAPADO,2020-04-20,C5/200408/01362,19.424605,-99.15883,AV CHAPULTEPEC,...,CRUZ,SI,EJE VIAL,P-O,Miércoles,MEDIA,RADIO,lesionada,NaN,CUAUHTEMOC
22,faltante,NaN,2020-04-08,13:42:00,DERRAPADO,2020-04-20,C5/200408/03206,19.41372,-99.136785,EJE 2 A SUR,...,CRUZ,SI,EJE VIAL,P-O,Miércoles,BAJA,911 CDMX,lesionada,NaN,CUAUHTEMOC
23,faltante,NaN,2020-04-08,17:35:00,DERRAPADO,2020-04-20,CH/200408/03536,19.434724,-99.160424,SERAPIO RENDON,...,CRUZ,SI,VIA SECUNDARIA,S-N,Miércoles,MEDIA,BOTON DE AUXILIO,lesionada,NaN,CUAUHTEMOC
24,faltante,NaN,2020-04-08,19:05:00,DERRAPADO,2020-04-20,CH/200408/05419,19.422969,-99.134178,TLAXCOAQUE,...,CURVA,NO,VIA SECUNDARIA,SO-NP,Miércoles,MEDIA,BOTON DE AUXILIO,lesionada,NaN,CUAUHTEMOC
31,faltante,NaN,2020-04-09,05:12:00,CHOQUE,2020-04-23,CH/200409/01064,19.425749,-99.13141,JOSE MA IZAZAGA,...,T,SI,EJE VIAL,P-O,Jueves,MEDIA,RADIO,lesionada,NaN,CUAUHTEMOC



Filas originales intactas: 158076
Casos marcados para revisión: 13932


## 12. Comprobaciones finales y reporte de impacto
Verificamos que las cinco tablas permanecen sin modificaciones: **la Etapa 2 es de diagnóstico**. Registramos un resumen consolidado y las rutas de evidencias. No sumamos filas de tablas distintas para obtener un número de hechos únicos.

In [12]:
resumen_final = (
    inventario_df[['tabla', 'filas', 'columnas']]
    .merge(validaciones_df.drop(columns=['filas']), on='tabla', how='left')
    .merge(duplicados_df.drop(columns=['filas']), on='tabla', how='left')
    .merge(folios_df, on='tabla', how='left')
)
resumen_final['filas_eliminadas_etapa2'] = 0
resumen_final.to_csv(DESTINO_ETAPA2 / 'resumen_final_validacion_SSC.csv',
                     index=False, encoding='utf-8-sig')
display(resumen_final)

# Releemos cada CSV para comprobar que no alteramos las tablas en memoria.
for ruta in ARCHIVOS:
    nombre = ruta.stem.replace('_etapa1', '')
    original = pd.read_csv(ruta, dtype=str, encoding='utf-8-sig', low_memory=False)
    assert TABLAS[nombre].equals(original), f'Se modificó la tabla {nombre}'
print('COMPROBACIÓN SUPERADA: no modificamos ni eliminamos filas de los cinco CSV.')
print('Evidencias guardadas en:', DESTINO_ETAPA2)
print('Archivos de evidencia:', len(list(DESTINO_ETAPA2.glob('*.csv'))))

,tabla,filas,columnas,fechas_invalidas,fechas_antes_2018,fechas_futuras_alerta,alcaldias_no_validas,anios_encontrados,filas_en_grupos_identicos,filas_repetidas_adicionales,filas_sin_repeticion_adicional,filas_eliminadas_en_etapa2,tiene_folio,folios_vacios,folios_sd_o_equivalentes,folios_distintos_utilizables,folios_utilizables_repetidos,filas_en_folios_repetidos,maximo_filas_por_folio,filas_eliminadas_etapa2
0,HechosTransito_SSC,134078,27,0,0,0,0,"2018, 2019, 2020, 2021, 2022, 2023",0,0,134078,0,True,1,7288,126470,218,537,5,0
1,PersonasLyFporEdad_SSC,158076,24,0,0,0,0,"2018, 2019, 2020, 2021, 2022, 2023",6486,3498,154578,0,True,0,8651,126251,16718,39892,25,0
2,PersonasLyFporSexo_SSC,144043,25,0,0,0,0,"2018, 2019, 2020, 2021, 2022, 2023",0,0,144043,0,True,1,7828,126470,9433,19177,6,0
3,PersonasLyFporTipoPersona_SSC,107794,25,0,0,0,0,"2020, 2021, 2022, 2023",0,0,107794,0,True,0,17,98998,8553,17332,5,0
4,VehiculosInvolucrados_SSC,215078,27,0,0,0,0,"2018, 2019, 2020, 2021, 2022, 2023",0,0,215078,0,True,2,11797,126470,72779,149588,9,0


COMPROBACIÓN SUPERADA: no modificamos ni eliminamos filas de los cinco CSV.
Evidencias guardadas en: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_SSC/etapa2
Archivos de evidencia: 18


In [13]:
from pathlib import Path
import pandas as pd

carpeta = Path(DESTINO_ETAPA2)

print("ARCHIVOS RELACIONADOS CON EDADES:")
for archivo in sorted(carpeta.glob("*edad*")):
    print(archivo.name)

print("\nRESUMEN DE VALIDACIÓN DE EDADES:")

archivo_resumen = carpeta / "resumen_validacion_edad_SSC.csv"

if archivo_resumen.exists():
    resumen_edad = pd.read_csv(archivo_resumen)
    display(resumen_edad)
else:
    print("No se encontró el resumen de edades.")

ARCHIVOS RELACIONADOS CON EDADES:
casos_edad_para_revision_SSC.csv
frecuencias_edades_superiores_120_SSC.csv
resumen_validacion_edad_SSC.csv

RESUMEN DE VALIDACIÓN DE EDADES:


,categoria,filas,porcentaje
0,dentro_rango_operativo,144144,91.19
1,faltante,13918,8.80
2,superior_a_120,14,0.01


Interpretación: de las 158,076 filas, 13,932 presentan una edad faltante o superior al límite operativo de 120 años.
Esto representa aproximadamente el 8.81% de los registros. Sin embargo, la mayoría de estos casos corresponde a edades faltantes, no a edades extremas.
Las 14 edades superiores a 120 años requieren revisión porque pueden deberse a errores de captura o codificación. Todavía no conocemos sus valores exactos ni podemos afirmar que todas correspondan a edades de 300 años.

## 13. Hallazgos y decisiones para la Etapa 3
**Nuestra interpretación se completa después de ejecutar las celdas anteriores:**

- Revisamos las columnas con más valores faltantes en `perfil_columnas_SSC.csv`.
- Distinguimos los folios `SD` y los folios repetidos de los duplicados exactos.
- Consultamos `resumen_duplicados_SSC.csv` y las muestras por tabla antes de proponer una eliminación.
- Conservamos los registros cuyo folio aparece en alcaldías diferentes, de acuerdo con la regla de revisión del equipo.
- No unimos las tablas de personas y vehículos como si sus filas fueran hechos únicos.
- Documentaremos cualquier corrección o exclusión de la Etapa 3 con criterio, evidencia y conteo antes/después.

**Conclusión preliminar:** esta etapa produce un diagnóstico reproducible de calidad y validación de SSC. No cambia los datos delimitados y deja pendientes de decisión los posibles duplicados, valores faltantes y casos de folios compartidos.

**Evidencias principales:** `inventario_etapa2_SSC.csv`, `perfil_columnas_SSC.csv`, `frecuencias_top10_SSC.csv`, `validacion_temporal_geografica_SSC.csv`, `resumen_folios_SSC.csv`, `resumen_duplicados_SSC.csv`, `folios_multiples_alcaldias_top100_SSC.csv`, `perfil_numerico_SSC.csv` y `resumen_final_validacion_SSC.csv`, `resumen_validacion_edad_SSC.csv`, `casos_edad_para_revision_SSC.csv` y `frecuencias_edades_superiores_120_SSC.csv`.

- Revisamos también `resumen_validacion_edad_SSC.csv`, `casos_edad_para_revision_SSC.csv` y `frecuencias_edades_superiores_120_SSC.csv` para justificar reglas de tratamiento de edades en la Etapa 3, sin corregirlas automáticamente.
